# UA-FCCL: export the contexts for ConExp (Colab)
Run the cells **one by one, from top to bottom**. If this Colab session was used before, first click **Runtime > Restart session**.

**Files needed on your computer:** `extra.py` and `cxt_export.py` (both sent in chat).

The notebook writes `.cxt` files (Burmeister format). In ConExp use **File > Open** and choose the `.cxt` file. Each row of the context is a group of training objects with the same attribute set; its name shows the group size and the majority class, for example `P3_n41_c2`. Cell 5 prints the number of concepts, which ConExp must show too.

## Cell 1: get the repo and one package

In [ ]:
%cd /content
!rm -rf conformal-fuzzy-ccl
!git clone -q https://github.com/mahibgm9676-lab/conformal-fuzzy-ccl.git
%cd /content/conformal-fuzzy-ccl
!pip -q install ucimlrepo
import os
os.environ['CCL_DING_DIR'] = '/content/class-conditional-conformal'
!ls ccl_helpers.py

## Cell 2: upload `extra.py` and `cxt_export.py`
Click **Choose Files** and select both files together.

In [ ]:
import glob, os, shutil
from google.colab import files
up = files.upload()
for base in ['extra', 'cxt_export']:
    c = sorted(glob.glob(base + '*.py'), key=os.path.getmtime)
    if c[-1] != base + '.py': shutil.copy(c[-1], base + '.py')
    print('using', c[-1])
!ls extra.py cxt_export.py

## Cell 3: download the datasets (ignore the Shill Bidding message)

In [ ]:
!python 01_prepare_data.py
!ls data

## Cell 4: write the context files (theta = 0.3, 0.5, 0.7 for Ecoli, Wilt, Yeast, HTRU2; split 0)

In [ ]:
import warnings; warnings.filterwarnings('ignore')
from cxt_export import export_cxt
os.makedirs('cxt', exist_ok=True)
rows = []
for n in ['ecoli', 'wilt', 'yeast', 'htru2']:
    for th in [0.3, 0.5, 0.7]:
        out = f'cxt/{n}_theta{th}.cxt'
        objs, attrs, full, clar = export_cxt(f'data/{n}.csv', th, 0, out)
        rows.append([n, th, objs, attrs, full, clar])
import pandas as pd
T = pd.DataFrame(rows, columns=['dataset', 'theta', 'object_groups', 'attributes', 'concepts_full', 'concepts_clarified'])
print(T.to_string(index=False))
T.to_csv('cxt/concept_counts.csv', index=False)

## Cell 5: download the files and send me the counts table

In [ ]:
!zip -q cxt_files.zip cxt/*
from google.colab import files
files.download('cxt_files.zip')